# Household data analyzer

Inspect **what is actually stored in SQLite**, including rows, categories, monthly totals, provenance and data quality. Default: `demo.sqlite`; change `DATABASE_NAME` below to inspect `budget.sqlite`.

The connection is **read-only**. This notebook does not import records, confirm coverage, change categories or call the LLM. Money calculations use `Decimal`; only chart coordinates use floats. Outputs may contain household data: clear all outputs before sharing or committing this notebook.


In [ ]:
from pathlib import Path
from decimal import Decimal
from collections import Counter
import json
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DATABASE_NAME = "demo.sqlite"  # Or "budget.sqlite"; never creates a missing database.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "budget" / "store.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook from the repository or notebooks directory.")
DB_PATH = (ROOT / DATABASE_NAME).resolve()
if not DB_PATH.is_file():
    raise FileNotFoundError(DB_PATH)
conn = sqlite3.connect(DB_PATH.as_uri() + "?mode=ro")
conn.execute("PRAGMA query_only = ON")
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 30)
print(f"Database: {DB_PATH} ({DB_PATH.stat().st_size:,} bytes)")
print("SQLite integrity:", conn.execute("PRAGMA integrity_check").fetchone()[0])


## Database inventory

Counts and schema distinguish transaction records from category mappings, coverage assertions, recurring payments, expected income and approved budgets. `demo_import`, when present, stores the original demo tables and reconciliation results; these totals are not transaction rows.


In [ ]:
tables = pd.read_sql_query(
    "SELECT name, sql FROM sqlite_master WHERE type='table' ORDER BY name", conn)
def quote_identifier(name):
    return '"' + name.replace('"', '""') + '"'
inventory = pd.DataFrame([
    {"table": name, "rows": conn.execute(
        f"SELECT COUNT(*) FROM {quote_identifier(name)}").fetchone()[0]}
    for name in tables["name"]
])
display(inventory)
display(tables)


## Transaction records

`month` is the reporting period. A missing `booking_date` means a monthly-only entry, not a transaction on the first day of that month. Amounts are positive magnitudes; `kind` determines income, expense, refund or transfer. EUR and TRY must remain separate.


In [ ]:
transactions = pd.read_sql_query("SELECT * FROM transactions ORDER BY month, id", conn)
transactions["amount_decimal"] = transactions["amount"].map(Decimal)
print(f"{len(transactions)} records")
display(transactions.drop(columns=["source", "amount_decimal"]).head(30))
print("Kinds:", transactions["kind"].value_counts().to_dict())
print("Currencies:", transactions["currency"].value_counts().to_dict())
print("Monthly-only rows:", int(transactions["booking_date"].isna().sum()))


## Monthly totals, with exact decimal arithmetic

Consumption = expenses minus refunds. BES rows in this demo were classified as assumed savings transfers and remain excluded from consumption. Income was not supplied for the unlabelled source table: a missing income total is shown as `None`, not zero. These are recorded consumption totals, not available bank cash or evidence of complete imports.


In [ ]:
def exact_sum(values):
    return sum(values, Decimal("0"))

monthly_rows = []
for (month, currency), rows in transactions.groupby(["month", "currency"], sort=True):
    def amount_for(kind):
        return exact_sum(rows.loc[rows["kind"].eq(kind), "amount_decimal"])
    income_present = rows["kind"].eq("income").any()
    monthly_rows.append({
        "month": month, "currency": currency, "records": len(rows),
        "expenses": amount_for("expense"), "refunds": amount_for("refund"),
        "consumption": amount_for("expense") - amount_for("refund"),
        "recorded_income": amount_for("income") if income_present else None,
        "transfers_including_BES": amount_for("transfer"),
        "card_repayments": amount_for("card_repayment"),
        "unresolved": amount_for("unresolved"),
    })
monthly = pd.DataFrame(monthly_rows, columns=["month", "currency", "records", "expenses", "refunds", "consumption", "recorded_income", "transfers_including_BES", "card_repayments", "unresolved"])
display(monthly)


## Category and monthly breakdown

Each record contributes once using its assigned category; category parent/group rows are not added as extra spending. Categories are proposals and demo rows carry `classification_review` tags. For example, `out` remains uncategorized and shopping group membership is editable.


In [ ]:
categories = pd.read_sql_query("SELECT * FROM categories ORDER BY id", conn)
groups = pd.read_sql_query('SELECT * FROM groups ORDER BY name, category', conn)
display(categories)
display(groups)
consumption_rows = transactions.loc[transactions["kind"].isin(["expense", "refund"])].copy()
consumption_rows["net"] = [a if k == "expense" else -a
    for a, k in zip(consumption_rows["amount_decimal"], consumption_rows["kind"])]
consumption_rows["category"] = consumption_rows["category"].fillna("uncategorized")
category_month = pd.DataFrame([
    {"currency": currency, "month": month, "category": category, "consumption": exact_sum(rows["net"])}
    for (currency, month, category), rows in consumption_rows.groupby(["currency", "month", "category"], sort=True)
], columns=["currency", "month", "category", "consumption"])
display(category_month)
for currency, rows in category_month.groupby("currency"):
    print(currency)
    display(rows.pivot(index="category", columns="month", values="consumption").fillna(Decimal("0")))


## Recorded spending charts

Charts show recorded rows only; gaps do not establish zero real spending. Conversion to float happens only for plotting, after exact calculations. Separate charts prevent mixing currencies.


In [ ]:
for currency, rows in monthly.groupby("currency"):
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.bar(rows["month"], rows["consumption"].map(float))
    ax.set(title=f"Recorded monthly consumption ({currency})", ylabel=currency, xlabel="Simulated month")
    plt.tight_layout()
    plt.show()

for currency, rows in category_month.groupby("currency"):
    totals = rows.groupby("category")["consumption"].agg(exact_sum).sort_values()
    fig, ax = plt.subplots(figsize=(9, max(3, len(totals) * 0.3)))
    ax.barh(totals.index, totals.map(float))
    ax.set(title=f"Recorded consumption by category ({currency})", xlabel=currency)
    plt.tight_layout()
    plt.show()


## Drill down into supporting rows

Edit these filters and rerun. This section displays all matching rows without changing the database. Amounts and names remain as recorded.


In [ ]:
MONTH = None                 # e.g. "2026-10"
CATEGORY = None              # e.g. "clothing"
PERSON = None                # e.g. "Menekşe"
CURRENCY = "EUR"
SEARCH = ""                  # e.g. "zalando"

selected = transactions.copy()
for column, value in [("month", MONTH), ("category", CATEGORY), ("person", PERSON), ("currency", CURRENCY)]:
    if value is not None:
        selected = selected.loc[selected[column].eq(value)]
if SEARCH:
    selected = selected.loc[selected["merchant"].str.contains(SEARCH, case=False, regex=False, na=False)]
print(f"{len(selected)} matching rows")
with pd.option_context("display.max_rows", None):
    display(selected[["id", "month", "booking_date", "merchant", "amount", "currency", "kind", "category", "person", "tags", "one_off"]])


## Provenance and original totals

The demo remapped source labels onto August–October 2026. Original labels, sections, row positions and values remain in each record's `source`. The first pasted table had no income. Subtotals are retained as metadata and reconciled, not inserted as purchases.


In [ ]:
def parse_source(value):
    try:
        return json.loads(value)
    except (TypeError, ValueError):
        return {"raw_source": value}
provenance = pd.DataFrame([
    {"id": row.id, "month": row.month, **parse_source(row.source)}
    for row in transactions.itertuples()
])
display(provenance.head(15))
if "demo_import" in set(tables["name"]):
    imports = pd.read_sql_query("SELECT * FROM demo_import", conn)
    reconciliation = []
    for record in imports.itertuples():
        for table in json.loads(record.reconciliation):
            for name, check in table["checks"].items():
                reconciliation.append({"month": table["month"], "original_label": table["original_label"],
                    "total_name": name, **check, "income_missing": table["income_missing"]})
    reconciliation = pd.DataFrame(reconciliation)
    display(reconciliation)
    if not reconciliation.empty:
        print("All original totals reconcile:", reconciliation["difference"].map(Decimal).eq(Decimal("0")).all())
else:
    print("No demo import metadata in this database.")


## Data quality and reporting limitations

Repeated merchants/amounts are candidates for inspection, not duplicates to delete. Separate legitimate purchases (including repeated Temu, Vodafone, Zalando or Eventim rows) must survive. No recurring payment is confirmed from merchant names alone.


In [ ]:
quality = {
    "missing_booking_date_monthly_only": int(transactions["booking_date"].isna().sum()),
    "person_unassigned": int(transactions["person"].isna().sum()),
    "uncategorized": int((transactions["category"].isna() | transactions["category"].eq("uncategorized")).sum()),
    "classification_review": int(transactions["tags"].str.contains("classification_review", regex=False, na=False).sum()),
    "unresolved_kind": int(transactions["kind"].eq("unresolved").sum()),
}
display(pd.DataFrame(quality.items(), columns=["check", "rows"]))
keys = ["month", "merchant", "amount", "currency", "kind"]
repeated = transactions.loc[transactions.duplicated(keys, keep=False)]
display(repeated[["id", *keys, "source"]])
for table_name in ["coverage", "recurring", "plans", "budgets"]:
    print(table_name)
    display(pd.read_sql_query(f"SELECT * FROM {quote_identifier(table_name)}", conn))
print("Missing coverage is not zero spending. Forecasts require confirmed coverage.")
print("Monthly-only entries cannot support daily month-end projections.")
print("No bank cash balance is inferred from consumption, transfers or BES contributions.")


## Reconcile the analysis itself

These checks verify the displayed category totals equal monthly consumption within each currency. They do not prove source completeness.


In [ ]:
for row in monthly.itertuples():
    category_total = exact_sum(category_month.loc[
        category_month["currency"].eq(row.currency) & category_month["month"].eq(row.month), "consumption"])
    assert category_total == row.consumption, (row.month, row.currency)
print("Category and monthly totals reconcile.")
conn.close()
